In [29]:
import boto3
import pandas as pd
import awswrangler as wr
import plotly.express as px
from datetime import datetime

In [30]:
session = boto3.Session(profile_name='hiv-project')
s3 = session.client('s3')
bucket_name = 'hiv-data-022784797781'

In [31]:
df_tables = wr.athena.read_sql_query(
    sql="select table_name from information_schema.tables where table_schema = 'hivdb_tce'",
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

df_tables

,table_name
0,dashboard
1,dim_regimen
2,dim_relative_time
3,dim_time
4,fact_measurements
5,fact_tce
6,fact_tce_old
7,mart_past_regimens
8,mart_yearly_summary
9,stg_measurements


In [32]:
sql = """
select
    regimen_signature,
    avg(baseline_log_rna) as avg_baseline_log_rna
from hivdb_tce.dashboard
where regimen_signature is not null
  and baseline_log_rna is not null
group by regimen_signature
order by avg_baseline_log_rna desc
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

fig = px.bar(
    df,
    x="avg_baseline_log_rna",
    y="regimen_signature",
    orientation="h"
)

fig.update_xaxes(
    range=[df["avg_baseline_log_rna"].min() - 0.1,
           df["avg_baseline_log_rna"].max() + 0.1]
)

fig.show()
fig.write_image("regimen_vs_rna.png")

In [37]:
sql = """
select
    regimen_signature,
    avg(baseline_cd4) as avg_baseline_cd4
from hivdb_tce.dashboard
where regimen_signature is not null
  and baseline_cd4 is not null
group by regimen_signature
order by avg_baseline_cd4 desc
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

fig = px.bar(
    df,
    x="avg_baseline_cd4",
    y="regimen_signature",
    orientation="h"
)

pad = (df["avg_baseline_cd4"].max() - df["avg_baseline_cd4"].min()) * 0.05

fig.update_xaxes(
    range=[
        df["avg_baseline_cd4"].min() - pad,
        df["avg_baseline_cd4"].max() + pad
    ]
)

fig.show()
fig.write_image("regimen_vs_cd4.png")

In [40]:
sql = """
select
    baseline_year,
    avg(baseline_cd4) as avg_baseline_cd4
from hivdb_tce.dashboard_unfiltered
where baseline_year is not null
  and baseline_cd4 is not null
group by baseline_year
order by baseline_year
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

fig = px.line(
    df,
    x="baseline_year",
    y="avg_baseline_cd4",
    markers=True
)

fig.show()
fig.write_image("baseline_cd4_vs_baseline_year_line.png")